In [7]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

# Add highlighting specifications
pp.clear_highlights()
pp.add_highlight(which='tags', style='gray')
pp.add_highlight(region='cre', which='upper', style='red')
pp.add_highlight(which='lower gap', style='bold blue')

pp.clear_pools()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA').named('template_pool')

mutated_pool = template_pool.mutagenize('cre',
                                        mutation_rate=0.1, 
                                        mode='random', 
                                        num_states=5, 
                                        seq_name_prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            seq_name_prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        op_iter_order=-1,
                        seq_name_prefix='site_').named('sites_pool')

insertion_pool = template_pool.insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential', 
                                              seq_name_prefix='ins_').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool])\
    .repeat_states(2, seq_name_prefix='v_', op_iter_order=-2)\
    .insert_kmers('bc', mode='random', length=5, seq_name_prefix='bc_')\
    .named('combo_pool')\

combo_pool.print_library(show_name=True,seed=10)

combo_pool: seq_length=None, num_states=40
state  seq
    0  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACtGG</cre>ATTACGG<bc>ttcat</bc>AGATCGGA
    1  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACtGG</cre>ATTACGG<bc>tgatg</bc>AGATCGGA
    2  TCCCGACT<cre>aGAAAGCGtGCAGTGcGCgCACAGG</cre>ATTACGG<bc>aacgc</bc>AGATCGGA
    3  TCCCGACT<cre>aGAAAGCGtGCAGTGcGCgCACAGG</cre>ATTACGG<bc>tacgt</bc>AGATCGGA
    4  TCCCGACT<cre>GGAAgaCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>atacg</bc>AGATCGGA
    5  TCCCGACT<cre>GGAAgaCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>gtttt</bc>AGATCGGA
    6  TCCCGACT<cre>tGAAAGCGGGCAGTGgGCACACtGG</cre>ATTACGG<bc>attat</bc>AGATCGGA
    7  TCCCGACT<cre>tGAAAGCGGGCAGTGgGCACACtGG</cre>ATTACGG<bc>gcact</bc>AGATCGGA
    8  TCCCGACT<cre>GGAAAGCGGGCAGcGAGCAaACAGG</cre>ATTACGG<bc>tactg</bc>AGATCGGA
    9  TCCCGACT<cre>GGAAAGCGGGCAGcGAGCAaACAGG</cre>ATTACGG<bc>catag</bc>AGATCGGA
   10  TCCCGACT<cre>-----GCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>gcgct</bc>AGATCGGA
   11  TCCCGACT<cre>-----GCGGGCAGTGAGCACACAGG</cre>ATTA

Pool(id=10, name='combo_pool', op='op[10]:get_kmers', num_states=40)

In [2]:
import statetracker as st
# Initialize manager
with st.Manager() as mgr:
    
    # Define leaf counters
    mut_state = st.State(5, name='mut_state')
    del_state = st.State(5, name='del_state')
    ins_site_state = st.State(2, name='ins_site_state')
    ins_position_state = st.State(5, name='ins_position_state')
    v_state = st.State(2).named('v_state')
    
    # Build composite counters
    ins_state = st.product([ins_position_state, ins_site_state],
        name='ins_state')
    cre_state = st.stack([mut_state, del_state, ins_state],
        name='cre_state')
    seq_state = st.product([cre_state, v_state], name='seq_state')
    
    bc_state = st.synced_to(seq_state).named('bc_state')

    # I want this 
    rows_state = st.State(6, name='rows_state')
    cols_state = st.State(8, name='cols_state')
    plate_state = st.product([rows_state, cols_state]).named('plate_state')
    
    st.sync(plate_state, seq_state)
    
# Print DAG
plate_state.print_dag('minimal')

# Print sequences
plate_state.get_iteration_df().reset_index()


plate_state (n=48)
├── bc_state (n=40)
├── seq_state (n=40)
│   └── [Product]
│       ├── cre_state (n=20)
│       │   └── [Stack]
│       │       ├── mut_state (n=5)
│       │       ├── del_state (n=5)
│       │       └── ins_state (n=10)
│       │           └── [Product]
│       │               ├── ins_position_state (n=5)
│       │               └── ins_site_state (n=2)
│       └── v_state (n=2)
└── [Product]
    ├── rows_state (n=6)
    └── cols_state (n=8)


,plate_state,bc_state,seq_state,cre_state,mut_state,del_state,ins_state,ins_position_state,ins_site_state,v_state,rows_state,cols_state
0,0,0,0,0,0,None,None,None,None,0,0,0
1,1,1,1,1,1,None,None,None,None,0,1,0
2,2,2,2,2,2,None,None,None,None,0,2,0
3,3,3,3,3,3,None,None,None,None,0,3,0
4,4,4,4,4,4,None,None,None,None,0,4,0
5,5,5,5,5,None,0,None,None,None,0,5,0
6,6,6,6,6,None,1,None,None,None,0,0,1
7,7,7,7,7,None,2,None,None,None,0,1,1
8,8,8,8,8,None,3,None,None,None,0,2,1
9,9,9,9,9,None,4,None,None,None,0,3,1
